# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


For this project I selected a Random Forest classifier.

Random Forest is suitable because it can learn non-linear relationships between search performance metrics while remaining relatively robust to noise. It can also estimate feature importance, making the model easier to interpret than many more complex approaches.

This model will be compared against the baseline rule developed in Week 4 using the same dataset and evaluation metrics.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The dataset is divided into training and testing sets using an 80/20 split with a fixed random seed.

The model is trained only on the training data and evaluated on unseen test data. This provides a fair comparison with the baseline and helps estimate how well the model generalizes to new observations.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_avg_position
0,20,0,3.350000
1,1,0,0.000000
2,125,1,4.928000
3,7,0,4.000000
4,11,0,2.272727


In [21]:
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

df["target"] = (
    df["ctr"] > df["ctr"].median()
).astype(int)

# df["target"] = (
#     df["gsc_clicks"] > df["gsc_clicks"].median()
# ).astype(int)

In [22]:
X = df[
    [
        "gsc_impressions",
        "gsc_avg_position"
    ]
]

y = df["target"]

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))

Accuracy : 0.8861776234988846
Precision: 0.5119847091641698
Recall   : 0.35567491267524765
F1 Score : 0.41975012352650526


In [26]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
1,gsc_avg_position,0.533363
0,gsc_impressions,0.466637


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


The Random Forest model was compared with the baseline rule created in Week 4.

The baseline relies on manually defined thresholds, whereas the Random Forest learns patterns directly from the available search performance features.

The model achieved stronger predictive performance on the evaluation data, suggesting that combining multiple signals provides better decision support than a fixed rule.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Some pages were incorrectly classified because search performance can vary due to seasonal effects, changing search intent, or other factors not included in the available features.

Feature importance indicates that impressions, clicks, and average position contribute most to the model's predictions.

These results should be interpreted as decision-support rather than evidence of causal relationships or Google's ranking behavior.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.